# 03 — MLflow Registry, Quality Gate Automatizado, Serving e Drift (PSI)
**Hipótese de Governança:** A promoção automatizada via *Quality Gate* contra o baseline heurístico, combinada à monitorização contínua de *drift* (PSI e KS) e auditoria de inferências, assegura confiabilidade estatística em produção e delimita quando o deslocamento de distribuição exige investigação ou retreinamento sob supervisão *Human-in-the-Loop*.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import settings
from src.pipeline.register import run_quality_gate_and_register
from src.monitoring.drift_detector import run_drift_audit

reg = run_quality_gate_and_register()
print("Resultado do Quality Gate e Registro MLflow:")
print(json.dumps(reg, indent=2, ensure_ascii=False, default=str))

## 1. Inspeção Estruturada do Quality Gate e Métricas de Promoção
Leitura tabular dos critérios avaliados no *Quality Gate* e verificação de consistência com os artefatos persistidos em `artifacts/`.

In [ ]:
if isinstance(reg, dict):
    display(pd.json_normalize(reg, sep=".").T.rename(columns={0: "valor"}))

qg_path = ROOT / "artifacts" / "quality_gate_report.json"
if qg_path.exists():
    qg_report = json.loads(qg_path.read_text(encoding="utf-8"))
    print("Chaves em quality_gate_report.json:", list(qg_report.keys()))

## 2. Auditoria de Drift Temporal (PSI e Estatística KS)
Avaliação da estabilidade das features entre janelas, destacando o impacto da reestruturação metodológica de 2025 e variáveis com *drift* severo ($	ext{PSI} \ge 0.25$).

In [ ]:
drift = run_drift_audit()
if isinstance(drift, dict) and "oot_2024_2025" in drift:
    oot = drift["oot_2024_2025"]
    print("Features com drift severo (2024 -> 2025):", oot.get("severe_features", []))
    if isinstance(oot.get("features"), dict):
        df_drift = pd.DataFrame.from_dict(oot["features"], orient="index")
        display(df_drift.sort_values(by=df_drift.columns[0], ascending=False).head(15) if len(df_drift) else df_drift)
else:
    print(json.dumps(drift, indent=2, ensure_ascii=False, default=str))

## 3. Rigor Operacional: Serving, Limitações e Human-in-the-Loop
* **Contratos de Serving e Auditoria:** A API (`FastAPI` + `Pydantic`) valida intervalos e tipos na entrada e regista eventos em `artifacts/inference_audit_log.jsonl`, permitindo rastreabilidade de predições.
* **Risco de Degradação Pós-Shift:** Quando ocorre *drift* severo em variáveis educacionais ou mudança de regra institucional, o escore probabilístico pode perder calibração mesmo que a arquitetura do modelo permaneça intacta.
* **Decisão Responsável:** O sistema atua como suporte à decisão psicopedagógica (*Human-in-the-Loop*), exigindo revisão humana e reavaliação periódica do modelo e do limiar.